# Análisis del dataset de empleados. Daniel

## Descripción del dataset

El archivo `dataset.csv` contiene información de empleados activos y combina datos de las tablas `HumanResources.Employee`, `Department`, `EmployeeDepartmentHistory`, `EmployeePayHistory` y `Shift` de SQL Server. La preparación y exportación inicial se realiza en `Connection.py`.

## Objetivo

Cargar el CSV generado por el proyecto, preparar sus tipos de datos y verificar su estructura e integridad antes de utilizarlo en análisis posteriores.

## Variables del dataset

| Variable | Descripción |
|---|---|
| `BusinessEntityID` | Identificador del empleado. |
| `JobTitle` | Cargo del empleado. |
| `BirthDate` | Fecha de nacimiento. |
| `Gender` | Género registrado. |
| `MaritalStatus` | Estado civil registrado. |
| `HireDate` | Fecha de contratación. |
| `SalariedFlag` | Indicador de salario fijo. |
| `VacationHours` | Horas de vacaciones disponibles. |
| `SickLeaveHours` | Horas de licencia por enfermedad disponibles. |
| `Departamento` | Nombre del departamento actual. |
| `GrupoDepartamento` | Grupo al que pertenece el departamento. |
| `Turno` | Nombre del turno actual. |
| `Tarifa` | Tarifa de pago más reciente disponible. |
| `PayFrequency` | Frecuencia de pago registrada. |

## Índice

A. [Librerías](#A.-Librerías)  
B. [Carga de datos y preparación del dataset](#B.-Carga-de-datos-y-preparación-del-dataset)

> **Antes de ejecutar:** instala las dependencias con `pip install -r requirements.txt`. Si `dataset.csv` no existe, configura las variables `DB_SERVER`, `DB_NAME`, `DB_USER` y `DB_PASSWORD` en el archivo local `.env` (no compartas ese archivo) y ejecuta `Connection.py` para generarlo. El notebook no contiene credenciales ni se conecta directamente a la base de datos.

## A. Librerías

- **pandas**: lectura, selección y preparación tabular del CSV.
- **SQLAlchemy**: creación del motor de conexión en `Connection.py`.
- **pyodbc**: controlador Python utilizado por SQLAlchemy para conectarse a SQL Server. También se debe instalar el controlador ODBC de SQL Server compatible en el equipo.

Las dependencias están declaradas en `requirements.txt`. La siguiente celda muestra las versiones instaladas sin imprimir información de conexión.

In [1]:
from pathlib import Path

import pandas as pd
import pyodbc
import sqlalchemy

print(f"pandas: {pd.__version__}")
print(f"SQLAlchemy: {sqlalchemy.__version__}")
print(f"pyodbc: {pyodbc.version}")

pandas: 3.0.6
SQLAlchemy: 2.1.4
pyodbc: 5.3.0


## B. Carga de datos y preparación del dataset

`Connection.py` consulta las tablas de SQL Server, conserva empleados activos, incorpora departamento, turno y tarifa más reciente, valida el resultado y lo guarda como `dataset.csv`. Aquí se carga ese resultado para mantener una sola implementación del proceso de extracción y joins.

Las fechas se interpretan como fechas y `Tarifa` se convierte a tipo numérico. Si el CSV todavía no se ha generado, la celda indica cómo crearlo.

In [2]:
dataset_path = Path("dataset.csv")
if not dataset_path.is_file():
    raise FileNotFoundError(
        "No se encontró dataset.csv. Configura el archivo .env y ejecuta "
        "Connection.py para extraer y preparar los datos desde SQL Server."
    )

columnas_esperadas = [
    "BusinessEntityID", "JobTitle", "BirthDate", "Gender", "MaritalStatus",
    "HireDate", "SalariedFlag", "VacationHours", "SickLeaveHours",
    "Departamento", "GrupoDepartamento", "Turno", "Tarifa", "PayFrequency",
]

dataset = pd.read_csv(
    dataset_path,
    encoding="utf-8-sig",
    parse_dates=["BirthDate", "HireDate"],
)

columnas_faltantes = sorted(set(columnas_esperadas) - set(dataset.columns))
if columnas_faltantes:
    raise ValueError(f"Faltan columnas requeridas en el CSV: {columnas_faltantes}")

dataset = dataset[columnas_esperadas].copy()
dataset["Tarifa"] = pd.to_numeric(dataset["Tarifa"], errors="raise")

print(f"Filas: {dataset.shape[0]:,}")
print(f"Columnas: {dataset.shape[1]}")
dataset.head()

Filas: 290
Columnas: 14


,BusinessEntityID,JobTitle,BirthDate,Gender,MaritalStatus,HireDate,SalariedFlag,VacationHours,SickLeaveHours,Departamento,GrupoDepartamento,Turno,Tarifa,PayFrequency
0,1,Chief Executive Officer,1969-01-29,M,S,2009-01-14,True,99,69,Executive,Executive General and Administration,Day,125.5000,2
1,2,Vice President of Engineering,1971-08-01,F,S,2008-01-31,True,1,20,Engineering,Research and Development,Day,63.4615,2
2,3,Engineering Manager,1974-11-12,M,M,2007-11-11,True,2,21,Engineering,Research and Development,Day,43.2692,2
3,4,Senior Tool Designer,1974-12-23,M,S,2007-12-05,False,48,80,Tool Design,Research and Development,Day,29.8462,2
4,5,Design Engineer,1952-09-27,F,M,2008-01-06,True,5,22,Engineering,Research and Development,Day,32.6923,2


### Verificación de integridad

Se comprueba que el identificador sea único y que no haya valores nulos. Si alguna condición no se cumple, la celda informa del problema en lugar de dar por válida la preparación.

In [3]:
duplicados = int(dataset["BusinessEntityID"].duplicated().sum())
nulos_por_columna = dataset.isna().sum()

if duplicados:
    raise ValueError(f"Se encontraron {duplicados} identificadores de empleado duplicados.")
if nulos_por_columna.any():
    raise ValueError(f"Se encontraron valores nulos:\n{nulos_por_columna[nulos_por_columna > 0]}")

print("Validación correcta: IDs únicos y sin valores nulos.")
print("Tipos de datos:")
display(dataset.dtypes.to_frame(name="Tipo"))
print("Resumen numérico:")
display(dataset.describe(include="all").transpose())

Validación correcta: IDs únicos y sin valores nulos.
Tipos de datos:


,Tipo
BusinessEntityID,int64
JobTitle,str
BirthDate,datetime64[us]
Gender,str
MaritalStatus,str
HireDate,datetime64[us]
SalariedFlag,bool
VacationHours,int64
SickLeaveHours,int64
Departamento,str


Resumen numérico:


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
BusinessEntityID,290.0,NaN,NaN,NaN,145.5,1.0,73.25,145.5,217.75,290.0,83.860002
JobTitle,290,67,Production Technician - WC60,26,NaN,NaN,NaN,NaN,NaN,NaN,NaN
BirthDate,290,NaN,NaN,NaN,1978-07-04 12:24:49.655172,1951-10-17 00:00:00,1973-09-21 18:00:00,1978-10-19 12:00:00,1986-05-27 12:00:00,1991-05-31 00:00:00,NaN
Gender,290,2,M,206,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MaritalStatus,290,2,M,146,NaN,NaN,NaN,NaN,NaN,NaN,NaN
HireDate,290,NaN,NaN,NaN,2009-05-19 05:57:31.034482,2006-06-30 00:00:00,2008-12-26 06:00:00,2009-02-02 00:00:00,2009-10-09 18:00:00,2013-05-30 00:00:00,NaN
SalariedFlag,290,2,False,238,NaN,NaN,NaN,NaN,NaN,NaN,NaN
VacationHours,290.0,NaN,NaN,NaN,50.613793,0.0,26.25,51.0,75.0,99.0,28.786215
SickLeaveHours,290.0,NaN,NaN,NaN,45.306897,20.0,33.0,46.0,58.0,80.0,14.540444
Departamento,290,16,Production,179,NaN,NaN,NaN,NaN,NaN,NaN,NaN
